# MLForecast pipeline reports

Reports are opt-in. The default `report_level="off"` skips all report instrumentation. Use `"basic"` for timing and structural information, or `"detailed"` when you also want dataframe-memory and process-RSS measurements.

In [1]:
import numpy as np
import pandas as pd
import polars as pl
from sklearn.linear_model import LinearRegression
from utilsforecast.losses import smape

from mlforecast import MLForecast

rng = np.random.default_rng(0)
steps = np.arange(10, dtype=np.float64)
noisy_target = 0.5 * steps + rng.normal(scale=1.5, size=len(steps))
pdf = pd.DataFrame({
    'unique_id': ['a'] * 10 + ['b'] * 10,
    'ds': list(range(10)) * 2,
    'y': np.tile(noisy_target, 2),
})
pldf = pl.from_pandas(pdf)

def make_fcst(**kwargs):
    return MLForecast(
        models=LinearRegression(), freq=1, lags=[1, 2], **kwargs
    )

## Reporting levels

`off` is the default and creates no report attributes. `basic` avoids the expensive memory/RSS probes; `detailed` enables them.

In [2]:
off = make_fcst()
off.fit(pdf)
print('reports created at default level:', hasattr(off, 'fit_report_'))

basic = make_fcst(report_level='basic')
basic.preprocess(pdf, return_X_y=True)
basic.feature_preparation_report_.to_dict()

reports created at default level: False


{'input_backend': 'pandas',
 'output_backend': 'pandas',
 'input_shape': (20, 3),
 'output_shape': (16, 2),
 'input_dtypes': {'unique_id': 'object', 'ds': 'int64', 'y': 'float64'},
 'output_dtypes': {'lag1': 'float64', 'lag2': 'float64'},
 'operations': ['received pandas input',
  'dropped 4 rows with unavailable features or targets',
  'kept pandas representation'],
 'input_memory_bytes': None,
 'output_memory_bytes': None,
 'elapsed_seconds': 0.011065584025345743,
 'input_memory_mb': None,
 'output_memory_mb': None}

### What `off` skips

`off` does **not** make forecasting a no-op: MLForecast still validates data, prepares features, fits models, and produces forecasts. It skips only report work: timers, process-RSS reads, dataframe deep-memory scans, report construction, and (unless explicitly requested through `model_metrics`) quality-metric evaluation. The next cell replaces those hooks with errors; its successful execution and zero call counts demonstrate that the default path does not invoke them.

In [3]:
from unittest.mock import patch

def forbidden_hook(*args, **kwargs):
    raise AssertionError('report instrumentation should be off')

with (
    patch('mlforecast.forecast.perf_counter', side_effect=forbidden_hook) as timer,
    patch('mlforecast.forecast.get_process_rss_bytes', side_effect=forbidden_hook) as rss,
    patch('mlforecast.model_report._memory_bytes', side_effect=forbidden_hook) as memory,
    patch('mlforecast.forecast.summarize_model_metrics', side_effect=forbidden_hook) as metrics,
):
    verified_off = make_fcst()  # report_level='off' and no model_metrics
    verified_off.fit(pdf)
    verified_off.predict(1)
    verified_off.cross_validation(pdf, n_windows=2, h=1)

{
    'perf_counter_calls': timer.call_count,
    'rss_calls': rss.call_count,
    'deep_memory_calls': memory.call_count,
    'metric_summary_calls': metrics.call_count,
    'report_attributes_created': [
        name
        for name in ('feature_preparation_report_', 'model_fit_report_', 'fit_report_', 'predict_report_')
        if hasattr(verified_off, name)
    ],
}

{'perf_counter_calls': 0,
 'rss_calls': 0,
 'deep_memory_calls': 0,
 'metric_summary_calls': 0,
 'report_attributes_created': []}

In [4]:
detailed = make_fcst(report_level='detailed')
X_np, _ = detailed.preprocess(pldf, return_X_y=True, as_numpy=True)
print(type(X_np), X_np.dtype, X_np.shape)
detailed.feature_preparation_report_.to_dict()

<class 'numpy.ndarray'> float64 (16, 2)


{'input_backend': 'polars',
 'output_backend': 'numpy',
 'input_shape': (20, 3),
 'output_shape': (16, 2),
 'input_dtypes': {'unique_id': 'String', 'ds': 'Int64', 'y': 'Float64'},
 'output_dtypes': {'feature_0': 'float64', 'feature_1': 'float64'},
 'operations': ['received polars input',
  'dropped 4 rows with unavailable features or targets',
  'converted polars -> numpy because as_numpy=True'],
 'input_memory_bytes': 340,
 'output_memory_bytes': 256,
 'elapsed_seconds': 0.1602872910152655,
 'input_memory_mb': 0.000324249267578125,
 'output_memory_mb': 0.000244140625}

## Fit, prediction, and report comparison

All report types expose `.diff(other)` for structured numeric comparisons. The candidate report calls `diff` with the baseline report.

In [5]:
basic.fit(pdf)
basic.predict(2)
detailed.fit(pdf)
detailed.predict(2)

comparison = detailed.fit_report_.diff(basic.fit_report_)
{
    'basic_fit': basic.fit_report_.to_dict(),
    'detailed_predict': detailed.predict_report_.to_dict(),
    'fit_diff': comparison.metrics,
}

{'basic_fit': {'elapsed_seconds': 0.01078675000462681,
  'rss_start_bytes': None,
  'rss_end_bytes': None,
  'rss_delta_bytes': None,
  'rss_delta_mb': None,
  'model_fit_report': {'elapsed_seconds': 0.0017824160167947412,
   'fit_calls': 1,
   'model_seconds': {'LinearRegression': 0.00176379201002419},
   'rss_start_bytes': None,
   'rss_end_bytes': None,
   'rss_delta_bytes': None,
   'rss_delta_mb': None},
  'calibration_model_fit_report': None,
  'final_model_fit_report': {'elapsed_seconds': 0.0017824160167947412,
   'fit_calls': 1,
   'model_seconds': {'LinearRegression': 0.00176379201002419},
   'rss_start_bytes': None,
   'rss_end_bytes': None,
   'rss_delta_bytes': None,
   'rss_delta_mb': None}},
 'detailed_predict': {'elapsed_seconds': 0.0050812080153264105,
  'rss_start_bytes': 253198336,
  'rss_end_bytes': 253198336,
  'horizon': 2,
  'rss_delta_bytes': 0,
  'rss_delta_mb': 0.0},
 'fit_diff': {'elapsed_seconds': MetricComparison(baseline=0.01078675000462681, candidate=0.015

## Quality metrics from actuals

The target has deterministic random noise, so the linear lag model cannot forecast every point perfectly. Quality metrics are explicitly opt-in: pass `model_metrics=[smape]` (or a list of compatible metric functions) when creating MLForecast. When fitted values or cross-validation actuals exist, MLForecast then records per-model metrics.

In [6]:
fitted = make_fcst(model_metrics=[smape])
fitted.fit(pdf, fitted=True)
fitted.model_metrics_

{'LinearRegression': {'smape': 0.15704198642059503}}

## Cross-validation reports

Each refit retains its own report instead of overwriting the previous fold's numbers. The returned holdout predictions also receive per-model quality metrics.

In [7]:
cv = make_fcst(report_level='basic', model_metrics=[smape])
cv_predictions = cv.cross_validation(pdf, n_windows=3, h=1)

{
    'predictions': cv_predictions,
    'fit_reports_per_refit': len(cv.cv_fit_reports_),
    'feature_reports_per_refit': len(cv.cv_feature_preparation_reports_),
    'model_fit_reports_per_refit': len(cv.cv_model_fit_reports_),
    'report_folds': cv.cv_report_folds_,
    'cv_model_metrics_pooled': cv.cv_model_metrics_,
    'cv_model_metrics_mean_by_fold': cv.cv_model_metrics_mean_,
    'cv_model_metrics_by_fold': cv.cv_model_metrics_by_fold_,
}

{'predictions':   unique_id  ds  cutoff         y  LinearRegression
 0         a   7       6  4.920621          5.442569
 1         b   7       6  4.920621          5.442569
 2         a   8       7  2.944397          4.737755
 3         b   8       7  2.944397          4.737755
 4         a   9       8  2.601868          1.633450
 5         b   9       8  2.601868          1.633450,
 'fit_reports_per_refit': 3,
 'feature_reports_per_refit': 3,
 'model_fit_reports_per_refit': 3,
 'report_folds': [0, 1, 2],
 'cv_model_metrics_pooled': {'LinearRegression': {'smape': 0.17082103496091328}},
 'cv_model_metrics_mean_by_fold': {'LinearRegression': {'smape': 0.17082103496091328}},
 'cv_model_metrics_by_fold': {0: {'LinearRegression': {'smape': 0.05036551366239601}},
  1: {'LinearRegression': {'smape': 0.2334447132905355}},
  2: {'LinearRegression': {'smape': 0.22865287792980832}}}}